# 01 — Processamento dos dados

Notebook ajustado para gerar a base final em nível de **envio de oferta**: `account_id + offer_id + offer_received_time`.

Essa granularidade preserva múltiplos envios da mesma oferta para o mesmo cliente e permite construir features históricas somente com informações anteriores ao recebimento da oferta.

## 1. Configuração

In [1]:
import os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col, coalesce, when, lit, to_date, datediff, array_contains,
    count, sum as spark_sum, min as spark_min, max as spark_max,
    row_number, lag
)

In [3]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("offer_personalization_data_processing")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/19 02:44:27 WARN Utils: Your hostname, jorel.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.7 instead (on interface en0)
26/05/19 02:44:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 02:45:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.0.2'

## 2. Leitura dos dados

In [4]:
RAW_PATH = "../data/raw"
PROCESSED_PATH = "../data/processed"

offers_path = f"{RAW_PATH}/offers.json"
customers_path = f"{RAW_PATH}/profile.json"
transactions_path = f"{RAW_PATH}/transactions.json"

offers = spark.read.option("multiline", "true").json(offers_path)
customers = spark.read.option("multiline", "true").json(customers_path)
transactions = spark.read.option("multiline", "true").json(transactions_path)

print("Offers:", offers.count())
print("Customers:", customers.count())
print("Transactions:", transactions.count())

Offers: 10
Customers: 17000
Transactions: 306534


## 3. Processamento de clientes

In [5]:
max_registered_date = (
    customers
    .agg(
        spark_max(to_date(col("registered_on"), "yyyyMMdd")).alias("max_registered_date")
    )
    .collect()[0]["max_registered_date"]
)

customers_clean = (
    customers
    .withColumn("registered_on", to_date(col("registered_on"), "yyyyMMdd"))
    .withColumn("account_age_days", datediff(lit(max_registered_date), col("registered_on")))
    .withColumn("gender", when(col("gender").isNull(), "unknown").otherwise(col("gender")))
    .withColumn(
        "age",
        when((col("age").isNull()) | (col("age") >= 100), None).otherwise(col("age"))
    )
)

## 4. Processamento de ofertas

In [6]:
offers_clean = (
    offers
    .withColumn("email", array_contains(col("channels"), "email").cast("int"))
    .withColumn("mobile", array_contains(col("channels"), "mobile").cast("int"))
    .withColumn("social", array_contains(col("channels"), "social").cast("int"))
    .withColumn("web", array_contains(col("channels"), "web").cast("int"))
    .withColumn("duration", col("duration").cast("double"))
    .withColumn("discount_value", col("discount_value").cast("double"))
    .withColumn("min_value", col("min_value").cast("double"))
)

## 5. Normalização dos eventos

In [7]:
transactions_clean = (
    transactions
    .withColumn(
        "offer_id",
        coalesce(
            col("value.offer_id"),
            col("value.`offer id`")
        )
    )
    .withColumn("amount", col("value.amount"))
    .withColumn("reward", col("value.reward"))
    .withColumn("time_since_test_start", col("time_since_test_start").cast("double"))
    .drop("value")
)

events_encoded = (
    transactions_clean
    .withColumn("offer_received", when(col("event") == "offer received", 1).otherwise(0))
    .withColumn("offer_viewed", when(col("event") == "offer viewed", 1).otherwise(0))
    .withColumn("offer_completed", when(col("event") == "offer completed", 1).otherwise(0))
    .withColumn("transaction", when(col("event") == "transaction", 1).otherwise(0))
)

## 6. Base de envios de oferta

A unidade analítica é `account_id + offer_id + offer_received_time`.

In [8]:
received_offers = (
    events_encoded
    .filter(col("event") == "offer received")
    .select(
        "account_id",
        "offer_id",
        col("time_since_test_start").alias("offer_received_time")
    )
)

In [9]:
received_offers = (
    received_offers.alias("r")
    .join(
        offers_clean.alias("o"),
        col("r.offer_id") == col("o.id"),
        how="left"
    )
    .drop(col("o.id"))
)

## 7. Reconstrução da jornada da oferta

In [10]:
view_events = (
    events_encoded
    .filter(col("event") == "offer viewed")
    .select(
        "account_id",
        "offer_id",
        col("time_since_test_start").alias("viewed_time")
    )
)

view_candidates = (
    received_offers
    .select("account_id", "offer_id", "offer_received_time", "duration")
    .alias("r")
    .join(
        view_events.alias("v"),
        (
            (col("r.account_id") == col("v.account_id")) &
            (col("r.offer_id") == col("v.offer_id")) &
            (col("v.viewed_time") >= col("r.offer_received_time")) &
            (col("v.viewed_time") <= col("r.offer_received_time") + col("r.duration"))
        ),
        how="left"
    )
)

view_window = (
    Window
    .partitionBy(col("r.account_id"), col("r.offer_id"), col("r.offer_received_time"))
    .orderBy(col("v.viewed_time"))
)

view_matches = (
    view_candidates
    .withColumn("rn", row_number().over(view_window))
    .filter(col("rn") == 1)
    .select(
        col("r.account_id").alias("account_id"),
        col("r.offer_id").alias("offer_id"),
        col("r.offer_received_time").alias("offer_received_time"),
        col("v.viewed_time").alias("offer_viewed_time")
    )
)

In [11]:
completed_events = (
    events_encoded
    .filter(col("event") == "offer completed")
    .select(
        "account_id",
        "offer_id",
        col("time_since_test_start").alias("completed_time"),
        "reward"
    )
)

completed_candidates = (
    received_offers
    .select("account_id", "offer_id", "offer_received_time", "duration")
    .alias("r")
    .join(
        completed_events.alias("c"),
        (
            (col("r.account_id") == col("c.account_id")) &
            (col("r.offer_id") == col("c.offer_id")) &
            (col("c.completed_time") >= col("r.offer_received_time")) &
            (col("c.completed_time") <= col("r.offer_received_time") + col("r.duration"))
        ),
        how="left"
    )
)

completed_window = (
    Window
    .partitionBy(col("r.account_id"), col("r.offer_id"), col("r.offer_received_time"))
    .orderBy(col("c.completed_time"))
)

completed_matches = (
    completed_candidates
    .withColumn("rn", row_number().over(completed_window))
    .filter(col("rn") == 1)
    .select(
        col("r.account_id").alias("account_id"),
        col("r.offer_id").alias("offer_id"),
        col("r.offer_received_time").alias("offer_received_time"),
        col("c.completed_time").alias("offer_completed_time"),
        col("c.reward").alias("reward")
    )
)

## 8. Transações durante a janela da oferta

In [12]:
transaction_events = (
    events_encoded
    .filter(col("event") == "transaction")
    .select(
        "account_id",
        col("time_since_test_start").alias("transaction_time"),
        "amount"
    )
)

offer_transaction_metrics = (
    received_offers
    .select("account_id", "offer_id", "offer_received_time", "duration")
    .alias("r")
    .join(
        transaction_events.alias("t"),
        (
            (col("r.account_id") == col("t.account_id")) &
            (col("t.transaction_time") >= col("r.offer_received_time")) &
            (col("t.transaction_time") <= col("r.offer_received_time") + col("r.duration"))
        ),
        how="left"
    )
    .groupBy(
        col("r.account_id").alias("account_id"),
        col("r.offer_id").alias("offer_id"),
        col("r.offer_received_time").alias("offer_received_time")
    )
    .agg(
        count("t.transaction_time").alias("transactions_during_offer"),
        spark_sum("t.amount").alias("amount_during_offer")
    )
    .fillna({
        "transactions_during_offer": 0,
        "amount_during_offer": 0
    })
)

## 9. Base cliente-oferta

In [13]:
customer_offer_dataset = (
    received_offers.alias("r")
    .join(view_matches.alias("v"), on=["account_id", "offer_id", "offer_received_time"], how="left")
    .join(completed_matches.alias("c"), on=["account_id", "offer_id", "offer_received_time"], how="left")
    .join(offer_transaction_metrics.alias("tx"), on=["account_id", "offer_id", "offer_received_time"], how="left")
    .withColumn("offer_received", lit(1))
    .withColumn("offer_viewed", when(col("offer_viewed_time").isNotNull(), 1).otherwise(0))
    .withColumn("offer_completed", when(col("offer_completed_time").isNotNull(), 1).otherwise(0))
    .withColumn("time_to_view", col("offer_viewed_time") - col("offer_received_time"))
    .withColumn("time_to_complete", col("offer_completed_time") - col("offer_received_time"))
    .withColumn("reward", coalesce(col("reward"), lit(0.0)))
    .fillna({
        "transactions_during_offer": 0,
        "amount_during_offer": 0
    })
)

In [14]:
customer_offer_dataset = (
    customer_offer_dataset
    .join(
        customers_clean,
        customer_offer_dataset.account_id == customers_clean.id,
        how="left"
    )
    .drop("id")
)

## 10. Features históricas pré-oferta

As features abaixo usam somente eventos com tempo estritamente menor que `offer_received_time`.

In [15]:
events_encoded_enriched = (
    events_encoded
    .join(
        offers_clean.select(
            col("id").alias("offer_id_join"),
            "offer_type"
        ),
        events_encoded.offer_id == col("offer_id_join"),
        how="left"
    )
    .drop("offer_id_join")
    .withColumn(
        "bogo_offer_completed",
        when(
            (col("event") == "offer completed") &
            (col("offer_type") == "bogo"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "discount_offer_completed",
        when(
            (col("event") == "offer completed") &
            (col("offer_type") == "discount"),
            1
        ).otherwise(0)
    )
)

events_history = events_encoded_enriched.select(
    "account_id",
    "time_since_test_start",
    "transaction",
    "amount",
    "offer_received",
    "offer_completed",
    "bogo_offer_completed",
    "discount_offer_completed"
)

In [16]:
historical_features = (
    customer_offer_dataset
    .select("account_id", "offer_id", "offer_received_time")
    .alias("r")
    .join(
        events_history.alias("h"),
        (
            (col("r.account_id") == col("h.account_id")) &
            (col("h.time_since_test_start") < col("r.offer_received_time"))
        ),
        how="left"
    )
    .groupBy(
        col("r.account_id").alias("account_id"),
        col("r.offer_id").alias("offer_id"),
        col("r.offer_received_time").alias("offer_received_time")
    )
    .agg(
        spark_sum("h.transaction").alias("transactions_before_offer"),
        spark_sum("h.amount").alias("amount_before_offer"),
        spark_sum("h.offer_received").alias("offers_received_before"),
        spark_sum("h.offer_completed").alias("offers_completed_before"),
        spark_sum("h.bogo_offer_completed").alias("bogo_offers_completed_before"),
        spark_sum("h.discount_offer_completed").alias("discount_offers_completed_before")
    )
    .withColumn(
        "avg_ticket_before_offer",
        when(
            col("transactions_before_offer") > 0,
            col("amount_before_offer") / col("transactions_before_offer")
        ).otherwise(0)
    )
    .withColumn(
        "completion_rate_before",
        when(
            col("offers_received_before") > 0,
            col("offers_completed_before") / col("offers_received_before")
        ).otherwise(0)
    )
    .fillna({
        "transactions_before_offer": 0,
        "amount_before_offer": 0,
        "offers_received_before": 0,
        "offers_completed_before": 0,
        "bogo_offers_completed_before": 0,
        "discount_offers_completed_before": 0,
        "avg_ticket_before_offer": 0,
        "completion_rate_before": 0
    })
)

In [17]:
customer_offer_final = (
    customer_offer_dataset
    .join(
        historical_features,
        on=["account_id", "offer_id", "offer_received_time"],
        how="left"
    )
)

## 11. Tempo para concluir a oferta anterior

In [18]:
offer_sequence_window = (
    Window
    .partitionBy("account_id")
    .orderBy("offer_received_time", "offer_id")
)

customer_offer_final = (
    customer_offer_final
    .withColumn(
        "previous_offer_time_to_complete",
        lag("time_to_complete").over(offer_sequence_window)
    )
    .fillna({
        "previous_offer_time_to_complete": 0,
        "transactions_before_offer": 0,
        "amount_before_offer": 0,
        "offers_received_before": 0,
        "offers_completed_before": 0,
        "bogo_offers_completed_before": 0,
        "discount_offers_completed_before": 0,
        "avg_ticket_before_offer": 0,
        "completion_rate_before": 0
    })
)

## 12. Seleção e ordenação final

In [19]:
customer_offer_final = customer_offer_final.select(
    "account_id",
    "offer_id",

    "offer_received_time",
    "offer_viewed_time",
    "offer_completed_time",

    "offer_received",
    "offer_viewed",
    "offer_completed",

    "time_to_view",
    "time_to_complete",
    "previous_offer_time_to_complete",

    "transactions_during_offer",
    "amount_during_offer",

    "reward",
    "discount_value",
    "min_value",
    "duration",
    "offer_type",

    "email",
    "mobile",
    "social",
    "web",

    "age",
    "gender",
    "credit_card_limit",
    "registered_on",
    "account_age_days",

    "transactions_before_offer",
    "amount_before_offer",
    "offers_received_before",
    "offers_completed_before",
    "bogo_offers_completed_before",
    "discount_offers_completed_before",
    "avg_ticket_before_offer",
    "completion_rate_before"
).orderBy(
    "account_id",
    "offer_received_time",
    "offer_id"
)

## 13. Validações

In [20]:
print("customers_clean:", customers_clean.count())
print("offers_clean:", offers_clean.count())
print("transactions_clean:", transactions_clean.count())
print("received_offers:", received_offers.count())
print("customer_offer_final:", customer_offer_final.count())

customers_clean: 17000
offers_clean: 10
transactions_clean: 306534
received_offers: 76277


customer_offer_final: 76277


In [21]:
customer_offer_final.groupBy("offer_type").count().show(truncate=False)
customer_offer_final.groupBy("offer_completed").count().show(truncate=False)

+-------------+-----+
|offer_type   |count|
+-------------+-----+
|discount     |30543|
|informational|15235|
|bogo         |30499|
+-------------+-----+



+---------------+-----+
|offer_completed|count|
+---------------+-----+
|1              |33631|
|0              |42646|
+---------------+-----+



Exemplo de jornada com cliente no dataset agregado VS transações

In [22]:
# sample_account = "78afa995795e4d85b5d9ceeca43f5fef"
sample_account = "0861b9ca31b741bb8b411b18f82d18f6"


(
    customer_offer_final
    .filter(col("account_id") == sample_account)
    .orderBy("offer_received_time", "offer_id")
    .show(truncate=False, n=100)
)

+--------------------------------+--------------------------------+-------------------+-----------------+--------------------+--------------+------------+---------------+------------+----------------+-------------------------------+-------------------------+-------------------+------+--------------+---------+--------+-------------+-----+------+------+---+---+------+-----------------+-------------+----------------+-------------------------+-------------------+----------------------+-----------------------+----------------------------+--------------------------------+-----------------------+----------------------+
|account_id                      |offer_id                        |offer_received_time|offer_viewed_time|offer_completed_time|offer_received|offer_viewed|offer_completed|time_to_view|time_to_complete|previous_offer_time_to_complete|transactions_during_offer|amount_during_offer|reward|discount_value|min_value|duration|offer_type   |email|mobile|social|web|age|gender|credit_card_

In [23]:
# sample_account = "78afa995795e4d85b5d9ceeca43f5fef"
sample_account = "0861b9ca31b741bb8b411b18f82d18f6"

(
    transactions
    .filter(col("account_id") == sample_account)
    # .orderBy("offer_received_time", "offer_id")
    .show(truncate=False, n=100)
)

+--------------------------------+---------------+---------------------+----------------------------------------------------+
|account_id                      |event          |time_since_test_start|value                                               |
+--------------------------------+---------------+---------------------+----------------------------------------------------+
|0861b9ca31b741bb8b411b18f82d18f6|offer received |0.0                  |{NULL, f19421c1d4aa40978ebb69ca19b0e20d, NULL, NULL}|
|0861b9ca31b741bb8b411b18f82d18f6|offer viewed   |0.0                  |{NULL, f19421c1d4aa40978ebb69ca19b0e20d, NULL, NULL}|
|0861b9ca31b741bb8b411b18f82d18f6|transaction    |1.5                  |{15.59, NULL, NULL, NULL}                           |
|0861b9ca31b741bb8b411b18f82d18f6|offer completed|1.5                  |{NULL, NULL, f19421c1d4aa40978ebb69ca19b0e20d, 5.0} |
|0861b9ca31b741bb8b411b18f82d18f6|offer received |7.0                  |{NULL, f19421c1d4aa40978ebb69ca19b0e20d, NULL,

## 14. Salvamento

In [24]:
customers_clean.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/customers_clean.parquet"
)

offers_clean.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/offers_clean.parquet"
)

transactions_clean.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/transactions_clean.parquet"
)

customer_offer_final.write.mode("overwrite").parquet(
    f"{PROCESSED_PATH}/customer_offer_transactions.parquet"
)

## Observações metodológicas

- A unidade final é o envio de uma oferta para um cliente.
- A mesma oferta enviada mais de uma vez ao mesmo cliente é preservada em linhas diferentes.
- Features históricas usam apenas eventos anteriores ao recebimento da oferta.
- Colunas como `offer_viewed`, `time_to_view`, `time_to_complete`, `transactions_during_offer` e `amount_during_offer` descrevem comportamento posterior ao envio e devem ser usadas com cautela na modelagem para evitar leakage.